# 🪔 Diwali Sales Analysis
## Exploratory Data Analysis — Customer Behavior & Revenue Insights

---

**Dataset:** 11,239 records | 16 States | 18 Product Categories | 15 Occupations

**Goal:** Identify key customer segments and purchasing patterns to optimize festive season marketing.

---

### Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Data Cleaning & Preprocessing](#2-cleaning)
3. [Exploratory Data Analysis](#3-eda)
   - 3.1 Gender Analysis
   - 3.2 Age Group Analysis
   - 3.3 State & Zone Analysis
   - 3.4 Occupation Analysis
   - 3.5 Product Category Analysis
   - 3.6 Marital Status Analysis
   - 3.7 Correlation Analysis
4. [Key Insights & Recommendations](#4-insights)

## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

# Colour palette
ORANGE = '#E8650A'
GOLD   = '#F5A623'
DARK   = '#1A1A2E'
COLORS = [ORANGE, GOLD, '#E74C3C', '#3498DB', '#2ECC71',
          '#9B59B6', '#1ABC9C', '#F39C12', '#E67E22', '#16A085']

print('Libraries loaded ✅')

In [ ]:
df_raw = pd.read_csv('../data/Diwali_Sales_Data.csv', encoding='latin1')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('Column info:')
df_raw.info()
print('\nNull counts:')
print(df_raw.isnull().sum())

## 2. Data Cleaning & Preprocessing <a id='2-cleaning'></a>

In [ ]:
df = df_raw.copy()

# Drop irrelevant columns
df = df.drop(columns=['Status', 'unnamed1'], errors='ignore')
print(f'Dropped Status & unnamed1 columns')

# Drop 12 rows with missing Amount
before = len(df)
df = df.dropna(subset=['Amount'])
print(f'Dropped {before - len(df)} rows with null Amount')

# Fix dtypes
df['Amount'] = df['Amount'].astype(int)
df['User_ID'] = df['User_ID'].astype(str)

# Readable labels
df['Gender_Label']         = df['Gender'].map({'F': 'Female', 'M': 'Male'})
df['Marital_Status_Label'] = df['Marital_Status'].map({0: 'Single', 1: 'Married'})

# Ordered age categories
age_order = ['0-17','18-25','26-35','36-45','46-50','51-55','55+']
df['Age Group'] = pd.Categorical(df['Age Group'], categories=age_order, ordered=True)

print(f'\n✅ Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Key Statistics ────────────────────────────────────────────────────────────
total_rev    = df['Amount'].sum()
total_orders = df['Orders'].sum()
total_cust   = df['User_ID'].nunique()
avg_order    = total_rev / total_orders

print('═' * 45)
print(f'  Total Revenue    : ₹{total_rev:,.0f}')
print(f'  Total Orders     : {total_orders:,}')
print(f'  Unique Customers : {total_cust:,}')
print(f'  Avg Order Value  : ₹{avg_order:,.2f}')
print('═' * 45)

## 3. Exploratory Data Analysis <a id='3-eda'></a>

### 3.1 Gender Analysis

In [ ]:
g_rev = df.groupby('Gender_Label')['Amount'].sum()
g_ord = df.groupby('Gender_Label')['Orders'].sum()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title, clr in zip(
    axes,
    [g_rev, g_ord],
    ['Revenue Share by Gender', 'Orders Share by Gender'],
    [[ORANGE, GOLD], ['#E74C3C', '#3498DB']]
):
    ax.pie(data.values, labels=data.index, autopct='%1.1f%%', colors=clr,
           startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
    ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle('Gender Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/01_gender.png', dpi=150, bbox_inches='tight')
plt.show()

print(g_rev.to_string())

### 3.2 Age Group Analysis

In [ ]:
age = df.groupby('Age Group', observed=True).agg(
    Amount=('Amount','sum'),
    Orders=('Orders','sum'),
    Customers=('User_ID','nunique')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(age['Age Group'], age['Amount'], color=COLORS[:len(age)], edgecolor='white')
axes[0].set_title('Revenue by Age Group', fontsize=12, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
axes[0].spines[['top','right']].set_visible(False)
for bar, val in zip(bars, age['Amount']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100000,
                 f'₹{val/1e6:.1f}M', ha='center', fontsize=8, fontweight='bold')

axes[1].bar(age['Age Group'], age['Orders'], color=COLORS[:len(age)], edgecolor='white')
axes[1].set_title('Orders by Age Group', fontsize=12, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Age Group Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/02_age_group.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 State & Zone Analysis

In [ ]:
# Top 10 States
state = df.groupby('State')['Amount'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(state.index[::-1], state.values[::-1], color=COLORS[:10], edgecolor='white')
ax.set_title('Top 10 States by Revenue', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/03_state.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Zone
zone = df.groupby('Zone').agg(Amount=('Amount','sum'), Orders=('Orders','sum')) \
         .sort_values('Amount', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(zone['Zone'], zone['Amount'], color=COLORS[:5], edgecolor='white')
axes[0].set_title('Revenue by Zone', fontsize=12, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
axes[0].spines[['top','right']].set_visible(False)

axes[1].bar(zone['Zone'], zone['Orders'], color=COLORS[1:6], edgecolor='white')
axes[1].set_title('Orders by Zone', fontsize=12, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Zone / Regional Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/04_zone.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Occupation Analysis

In [ ]:
occ = df.groupby('Occupation')['Amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(occ.index, occ.values, color=COLORS[:len(occ)], edgecolor='white')
ax.set_title('Revenue by Occupation', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.set_xticklabels(occ.index, rotation=35, ha='right')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../charts/05_occupation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 Occupations:')
print(occ.head().apply(lambda x: f'₹{x:,.0f}'))

### 3.5 Product Category Analysis

In [ ]:
cat = df.groupby('Product_Category')['Amount'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(cat.index[::-1], cat.values[::-1], color=COLORS[:10], edgecolor='white')
ax.set_title('Top 10 Product Categories by Revenue', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(ax.patches, cat.values[::-1]):
    ax.text(bar.get_width()+50000, bar.get_y()+bar.get_height()/2,
            f'₹{val/1e6:.1f}M', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/06_product_category.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.6 Marital Status Analysis

In [ ]:
mar = df.groupby(['Marital_Status_Label','Gender_Label'])['Amount'].sum().unstack()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mar.plot(kind='bar', ax=axes[0], color=[ORANGE, GOLD], edgecolor='white', width=0.6)
axes[0].set_title('Revenue by Marital Status & Gender', fontsize=12, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e6:.1f}M'))
axes[0].set_xticklabels(mar.index, rotation=0)
axes[0].spines[['top','right']].set_visible(False)

mar_c = df.groupby(['Marital_Status_Label','Gender_Label'])['User_ID'].nunique().unstack()
mar_c.plot(kind='bar', ax=axes[1], color=['#E74C3C','#3498DB'], edgecolor='white', width=0.6)
axes[1].set_title('Customers by Marital Status & Gender', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(mar_c.index, rotation=0)
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Marital Status Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/07_marital_status.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.7 Correlation Analysis

In [ ]:
corr = df[['Age','Marital_Status','Orders','Amount']].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, annot_kws={'size': 12})
ax.set_title('Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/08_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(corr.round(3))

## 4. Key Insights & Recommendations <a id='4-insights'></a>

---

### 🔑 Key Findings

| # | Dimension | Finding |
|---|-----------|----------|
| 1 | **Gender** | Female buyers drive **~70%** of total revenue |
| 2 | **Age** | **26–35** age group is the top revenue contributor |
| 3 | **State** | **Uttar Pradesh** leads all 16 states in revenue |
| 4 | **Zone** | **Central zone** is the highest-grossing region |
| 5 | **Occupation** | **IT Sector** professionals spend the most |
| 6 | **Category** | **Food** is the best-selling product category |
| 7 | **Marital Status** | **Married women** are the highest-value customer segment |

---

### 🎯 Strategic Recommendations

1. **Primary Target Segment** → Married women aged 26–35 in IT/Healthcare in Western & Central India
2. **Geographic Focus** → Double down on Uttar Pradesh, Maharashtra, Karnataka
3. **Product Priority** → Feature Food, Clothing & Electronics as hero categories
4. **Channel Strategy** → Instagram & WhatsApp for females 26–35; LinkedIn for IT/Banking
5. **Loyalty Programs** → Build repeat-purchase incentives for 36–45 age group
6. **Regional Expansion** → Growth campaigns in Eastern zone — currently under-penetrated

---

*🪔 Report generated with Python | pandas | matplotlib | seaborn*